In [11]:
# IMPORTSSSS
import os
import glob
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout, Concatenate, RepeatVector, TimeDistributed, Bidirectional, LSTM


In [ ]:
# Hyperparameters
MAX_TIMESTEPS = 500  #  standardize all writing samples to 500 time-steps

# Delta_X (Horizontal stroke trajectory)
# Delta_Y (Vertical stroke trajectory)
# Pressure (Stress/Force)
# Tilt_X (Pen grip)
# Tilt_Y (Pen grip)
# Velocity (Speed)
# Acceleration (Momentum)
# Jerk (Smoothness/Micro-stutters)
FEATURES = 8         

In [13]:

# Preprocess blocks
def analyze_stroke_data(csv_filepath):
    # 1. Load the data
    df = pd.read_csv(csv_filepath)
    
    # 2. Calculate time differences between rows (Delta Time / dt)
    df['dt'] = df['time'].diff().fillna(0)
    
    # 3. Calculate distance between points (Delta Distance using Pythagorean theorem)
    df['dx'] = df['x'].diff().fillna(0)
    df['dy'] = df['y'].diff().fillna(0)
    
    # 4. Calculate Velocity (Distance / Time)
    # np.where prevents division-by-zero errors if two events fire at the exact same millisecond
    df['distance'] = np.sqrt(df['dx']**2 + df['dy']**2) 
    df['velocity'] = np.where(df['dt'] > 0, df['distance'] / df['dt'], 0)
    
    # Calculate "Writing Duration" 
    # Sum of the 'dt' column, but ONLY for rows where 'touching' == 1
    writing_duration = df['dt'].where(df['touching'] == 1, 0).sum()

    # Calculate "In-Air Pen Duration" (The pause time biomarker)
    # Sum of the 'dt' column, but ONLY for rows where 'touching' == 0
    in_air_duration = df['dt'].where(df['touching'] == 0, 0).sum()
    
    # TODO 3: Print the results!
    print(f"--- Analysis for: {csv_filepath} ---")
    print(f"Total Writing Duration : {writing_duration} ms")
    print(f"Total In-Air Pauses    : {in_air_duration} ms")
    print(f"Average Pen Velocity   : {df['velocity'].mean():.2f} px/ms")
    print("-" * 40)
    
    return df


In [ ]:
# PREPROCESSING and FEATURE EXTRACTION
def extract_golden_features(df):
    """Converts the raw CSV dataframe into the Golden 8 Features array."""
    # Safety checks for old files
    if "touching" not in df.columns: df["touching"] = True
    for col in ["tiltX", "tiltY", "latency"]:
        if col not in df.columns: df[col] = 0
        
    # Calculate the physics features
    df["dt"] = df["time"].diff().fillna(1)
    df.loc[df["dt"] == 0, "dt"] = 1
    
    df["delta_x"] = df["x"].diff().fillna(0)
    df["delta_y"] = df["y"].diff().fillna(0)
    df["distance"] = np.sqrt(df["delta_x"]**2 + df["delta_y"]**2)
    df["velocity"] = df["distance"] / df["dt"]
    df["acceleration"] = df["velocity"].diff().fillna(0) / df["dt"]
    df["jerk"] = df["acceleration"].diff().fillna(0) / df["dt"]
    
    # Features ARRAY
    golden_df = df[[
        "delta_x", 
        "delta_y", 
        "pressure", 
        "tiltX", 
        "tiltY", 
        "velocity", 
        "acceleration", 
        "jerk",
        ]]
    
    # Fill any weird math errors (like dividing by zero) with 0
    golden_df = golden_df.fillna(0)
    return golden_df.values

In [15]:


    
    
def load_and_pad_data(data_dir="datasets/"):
    sequences = []
    latencies = []
    labels = []
    
    csv_files = glob.glob(os.path.join(data_dir, "*.csv"))
    
    for file in csv_files:
        df = pd.read_csv(file)
        if len(df) == 0: continue
            
        # 1. Get Latency (from the very first row)
        latency_val = df["latency"].iloc[0] if "latency" in df.columns else 0
        
        # 2. Determine Label by reading the filename
        filename = os.path.basename(file).lower()
        if "normal" in filename:
            label = 0
        elif "dyslexia" in filename:
            label = 1
        else:
            continue # skip files that aren't labeled normal/dyslexia
            
        # 3. Use our new extractor! (Turns raw data into N x 8 array)
        stroke_data = extract_golden_features(df)
        
        # 4. Pad or Truncate to MAX_TIMESTEPS (500)
        if len(stroke_data) > MAX_TIMESTEPS:
            stroke_data = stroke_data[:MAX_TIMESTEPS] # Chop off excess
        else:
            padding = np.zeros((MAX_TIMESTEPS - len(stroke_data), FEATURES))
            stroke_data = np.vstack((stroke_data, padding)) # Add zeros to the end
            
        sequences.append(stroke_data)
        latencies.append(latency_val)
        # Expand the label to (500, 1)         
        labels.append(np.full((MAX_TIMESTEPS, 1), label))
        
    return np.array(sequences), np.array(latencies), np.array(labels)


In [16]:


def build_model():
    # kinematic sequence input
    sequence_input = Input(shape=(MAX_TIMESTEPS, FEATURES), name="kinematic_input")
    # Bidirectional reads the sequence forwards and backwards to understand context
    x = Bidirectional(LSTM(64, return_sequences=True))(sequence_input)
    x = Dropout(0.3)(x)
    x = Bidirectional(LSTM(32, return_sequences=True))(x)
    x = Dropout(0.3)(x)
    
    # latency input
    latency_input = Input(shape=(1,), name="latency_input")
    # Stretch the single Latency number across all 500 timesteps so it matches Branch A
    lat_repeated = RepeatVector(MAX_TIMESTEPS)(latency_input)
    
    # merge the inputs
    merged = Concatenate()([x, lat_repeated])
    output = TimeDistributed(Dense(1, activation='sigmoid'), name="heatmap_output")(merged)
    # Build and Compile
    
    model = Model(inputs=[sequence_input, latency_input], outputs=output, name='eldislexiav3',)
    model.compile(
        
        optimizer='adam', 
        loss='binary_crossentropy', 
        metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
    )
    return model

In [17]:
model = build_model()
model.summary()


Model: "eldislexiav3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ kinematic_input     │ (None, 500, 8)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 500, 128)  │     37,376 │ kinematic_input[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 500, 128)  │          0 │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 500, 64)   │     41,216 │ dropout_2[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ latency_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 500, 64)   │          0 │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_1     │ (None, 500, 1)    │          0 │ latency_input[0]… │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 500, 65)   │          0 │ dropout_3[0][0],  │
│ (Concatenate)       │                   │            │ repeat_vector_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ heatmap_output      │ (None, 500, 1)    │         66 │ concatenate_1[0]… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 78,658 (307.26 KB)

 Trainable params: 78,658 (307.26 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
print("Loading data...")
# Point this to your data collector folder
X_seq, X_lat, y = load_and_pad_data("datasets/") 
print(f"Data loaded! \nShape of X_seq (timeseries sequence) : {X_seq.shape} \nShape of X_lat (latency) :{X_lat.shape} \nShape of y (labels) : {y.shape}")
    

Loading data...
Data loaded! 
Shape of X_seq (timeseries sequence) : (302, 500, 8) 
Shape of X_lat (latency) :(302,) 
Shape of y (labels) : (302, 500, 1)


In [19]:

# Count how many total 0s and 1s exist in the entire training set 'y'
total_zeros = np.sum(y == 0)
total_ones = np.sum(y == 1)
total_samples = total_zeros + total_ones

# Apply the 'Weight = Total_Samples / (Number_of_Classes * Samples_in_Class)' formula
weight_for_0 = total_samples / (2.0 * total_zeros)
weight_for_1 = total_samples / (2.0 * total_ones)

# 3. Create the exact dictionary
sample_weight = np.ones(shape=y.shape) * weight_for_0
sample_weight[y == 1] = weight_for_1

print(f"Calculated Weights -> 0 (normal): {weight_for_0:.2f}, 1 (dyslexic): {weight_for_1:.2f}")

Calculated Weights -> 0 (normal): 0.93, 1 (dyslexic): 1.08


In [20]:
import numpy as np

print("Original X_seq Max Value:", np.max(X_seq))

# 1. Kill any "Infinity" values (caused by divide-by-zero physics errors)
X_seq_clean = np.nan_to_num(X_seq, nan=0.0, posinf=0.0, neginf=0.0)

# 2. Standardize the Time-Series Data (Z-Score Normalization)
# This finds the average of each of the 8 features and scales them so they hover around 0
feature_means = np.mean(X_seq_clean, axis=(0, 1))
feature_stds = np.std(X_seq_clean, axis=(0, 1))
feature_stds[feature_stds == 0] = 1 # Prevent division by zero crash

X_seq_scaled = (X_seq_clean - feature_means) / feature_stds

# 3. Scale Latency (Convert massive milliseconds into small Seconds, e.g. 3500ms -> 3.5s)
X_lat_scaled = X_lat / 1000.0

print("New X_seq_scaled Max Value:", np.max(X_seq_scaled))
print("Data successfully scaled! Ready for training.")

Original X_seq Max Value: 359.6499938964844
New X_seq_scaled Max Value: 98.09446571291596
Data successfully scaled! Ready for training.


In [21]:

print("Starting training...")
history = model.fit(
    [X_seq_scaled, X_lat_scaled], 
    y, 
    epochs=20, 
    validation_split=0.2, 
    sample_weight=sample_weight
    )
print("Saving the model...")
model.save("models/elkinematicV3.keras")

Starting training...
Epoch 1/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 36s 2s/step - accuracy: 0.6221 - loss: 0.8090 - recall: 0.1952 - val_accuracy: 0.5950 - val_loss: 0.8334 - val_recall: 0.3489
Epoch 2/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.6815 - loss: 0.7208 - recall: 0.3514 - val_accuracy: 0.6267 - val_loss: 0.7284 - val_recall: 0.3621
Epoch 3/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.6573 - loss: 0.6740 - recall: 0.3665 - val_accuracy: 0.6619 - val_loss: 0.6758 - val_recall: 0.4303
Epoch 4/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.8240 - loss: 0.5373 - recall: 0.7045 - val_accuracy: 0.9259 - val_loss: 0.3293 - val_recall: 0.8963
Epoch 5/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step - accuracy: 0.9308 - loss: 0.3145 - recall: 0.9240 - val_accuracy: 0.9390 - val_loss: 0.2646 - val_recall: 0.8959
Epoch 6/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step - accuracy: 0.9336 - loss: 0.2736 - recall: 0.9121 - val_accuracy: 0.9400 - val_loss: 0.2570 - val_recall: 0.8960
Epoch 7